# Fine-tune Wav2Vec2 (Thai XLS-R) สำหรับเสียงภาษาไทยถิ่น 3 ภาค (ใต้/เหนือ/อีสาน)

Notebook นี้ใช้ fine-tune โมเดล [`airesearch/wav2vec2-large-xlsr-53-th`](https://huggingface.co/airesearch/wav2vec2-large-xlsr-53-th) ให้รู้จำเสียงภาษาไทยถิ่น **ใต้ + เหนือ + อีสาน** ในโมเดลเดียว แล้วแปลงเป็นข้อความภาษาไทยกลาง

## 📋 สิ่งที่ Notebook นี้ทำ

```
ไฟล์เสียง .wav  ─→  Wav2Vec2 fine-tuned  ─→  ข้อความภาษาถิ่น  ─→  Lookup table  ─→  ข้อความภาษากลาง
                    (multi-dialect)         (south/north/isan)
```

## ⚙️ ก่อนเริ่ม

1. ตั้ง runtime เป็น **GPU**: `Runtime → Change runtime type → T4 GPU` (แนะนำ A100 ถ้ามี Colab Pro)
2. อัพโหลดโฟลเดอร์ `Data_Project` ทั้งหมดขึ้น Google Drive (~3–5 GB หลังเพิ่ม north + isan)
3. แก้ตัวแปร `DRIVE_PATH` ในเซลล์ 1.2 ให้ตรงกับ path ใน Drive

## ⏱️ เวลาที่ใช้ (โดยประมาณ) — หลังเพิ่ม north + isan

| ขั้นตอน | T4 (Colab ฟรี) | A100/V100 (Pro) |
|---|---|---|
| Setup + Build manifest + Vocab | 10–15 นาที | 10 นาที |
| Sanity check (3 epochs, originals only) | 10–15 นาที | 5 นาที |
| **Full training (8 epochs, ~120k clips, batch 16)** | **5–8 ชั่วโมง** | 2–3 ชั่วโมง |
| Inference + Eval (test ~15k clips) | 30–40 นาที | 10–15 นาที |

## 🛡️ Features ป้องกันงานเสีย (เพิ่มใหม่)

- **Checkpoint เก็บบน Google Drive** — ไม่หายแม้ Colab disconnect
- **Auto-resume** — รัน Cell 3.2 ใหม่ จะ resume จาก checkpoint ล่าสุดอัตโนมัติ
- **Drive cache สำหรับ dataset** — รอบหน้าหลัง disconnect copy จาก Drive (เร็วกว่า extract zip มาก)
- **Early stopping** — หยุดเองถ้า CER ไม่ดีขึ้น 3 evals ติด (ไม่เสียเวลาเทรนเปล่า)
- **Clean progress display** — แสดง %, ETA, loss trend, best CER แบบบรรทัดเดียว

---


# 🔧 ส่วนที่ 1: ตั้งค่า Environment

## Cell 1.1 — Mount Google Drive

เชื่อม Google Drive เข้ากับ Colab เพื่อเข้าถึงไฟล์ dataset

**สิ่งที่จะเกิดขึ้น:** ป๊อปอัพถามให้ login Google และให้สิทธิ์เข้าถึง Drive — กด **Connect**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Cell 1.2 — กำหนด Path และเตรียม Data บน Colab Disk

⚠️ **ต้องแก้:** ตั้ง `DRIVE_PATH` ให้ตรงกับสิ่งที่อัพไว้ใน Drive

Cell นี้ **auto-detect** ได้ทั้ง 2 แบบ:

| รูปแบบ | DRIVE_PATH | คำแนะนำ |
|---|---|---|
| 📁 **โฟลเดอร์** (audio_data + colab) | `/content/drive/MyDrive/Data_Project` | ง่าย แต่ upload ช้า (มีไฟล์เยอะ) |
| 🗜️ **ไฟล์ zip** (แนะนำ) | `/content/drive/MyDrive/Data_Project.zip` | upload เร็วกว่า 10–30 เท่า |

### 💾 Drive Cache (auto, ใหม่)

ครั้งแรกที่รัน:
1. Extract zip ลง `/content/Data_Project/` ตามปกติ
2. **Copy ต่อไปยัง `/content/drive/MyDrive/Data_Project_extracted/` ด้วย** (~15-30 นาทีเพิ่ม)

ครั้งถัดไป (หลัง disconnect):
- Cell นี้ตรวจเจอ cache → **copy จาก Drive cache ตรงๆ** (เร็วกว่า extract zip ~5-10 เท่า)
- ไม่ต้องแก้อะไร — ตั้ง `USE_CACHE = False` ใน cell ถ้าไม่ต้องการใช้

ใช้พื้นที่ Drive เพิ่ม ~3-5 GB

### โครงสร้างที่ Cell นี้คาดหวัง

ไม่ว่า DRIVE_PATH จะเป็นโฟลเดอร์หรือ zip ภายในต้องมี:
```
<root>/
├── audio_data/     ← จำเป็น (ไฟล์เสียง)
└── colab/          ← จำเป็น (scripts + notebook)
```
ส่วน `manifests/` และ `models/` ไม่ต้องอัพ — script จะสร้างให้เอง

### ทำไมต้อง copy/extract มาที่ Colab disk?

Colab อ่านไฟล์จาก Drive ช้ามาก (ช้ากว่า local disk ~10 เท่า) การคัดลอกหรือแตก zip มาที่ `/content/` ก่อนทำให้เทรนเร็วขึ้นมาก


In [ ]:
# ⚠️ แก้ DRIVE_PATH ให้ตรงกับสิ่งที่อัพไว้ใน Drive
# - ถ้าอัพเป็น folder: '/content/drive/MyDrive/Data_Project'
# - ถ้าอัพเป็น zip   : '/content/drive/MyDrive/Data_Project.zip'
DRIVE_PATH = '/content/drive/MyDrive/Data_Project.zip'

# โฟลเดอร์ทำงานบน Colab disk (เร็วกว่า Drive มาก)
WORK_DIR = '/content/Data_Project'

# 💾 (ใหม่) cache dataset ที่ extract แล้วไว้บน Drive — รอบหน้าจะ copy เร็วกว่า extract zip มาก
DRIVE_CACHE = '/content/drive/MyDrive/Data_Project_extracted'
USE_CACHE = True  # ตั้ง False ถ้าไม่อยากให้ใช้ Drive เป็น cache (จะ extract ใหม่ทุกครั้ง)

import os, shutil, zipfile, time

def _find_project_root(start_dir):
    """หาโฟลเดอร์ที่มี audio_data/ + colab/ อยู่ภายใน (สำหรับกรณี zip มี nested folder)"""
    from pathlib import Path
    root = Path(start_dir)
    if (root / 'audio_data').is_dir() and (root / 'colab').is_dir():
        return str(root)
    for sub in root.iterdir():
        if sub.is_dir() and (sub / 'audio_data').is_dir() and (sub / 'colab').is_dir():
            return str(sub)
    return None

if os.path.exists(WORK_DIR) and os.path.isdir(WORK_DIR) and os.listdir(WORK_DIR):
    print(f'✅ Dataset อยู่ที่ {WORK_DIR} แล้ว (ข้าม copy/extract)')
elif USE_CACHE and os.path.isdir(DRIVE_CACHE) and _find_project_root(DRIVE_CACHE):
    # ── Drive cache มีอยู่แล้ว: copy จาก Drive (เร็วกว่า extract zip) ─────
    print(f'⚡ พบ extracted cache บน Drive — copy {DRIVE_CACHE} → {WORK_DIR}')
    t0 = time.time()
    shutil.copytree(DRIVE_CACHE, WORK_DIR)
    print(f'✅ Copy จาก cache เสร็จใน {(time.time()-t0)/60:.1f} นาที')
elif DRIVE_PATH.endswith('.zip'):
    # ── โหมด zip: extract มาที่ Colab disk ────────────────────────────────
    if not os.path.exists(DRIVE_PATH):
        raise FileNotFoundError(f'ไม่พบไฟล์ zip ที่ {DRIVE_PATH} — แก้ DRIVE_PATH ให้ถูก')
    size_mb = os.path.getsize(DRIVE_PATH) / 1e6
    print(f'🗜️  ตรวจพบไฟล์ zip ({size_mb:,.0f} MB) — กำลังแตก ...')
    t0 = time.time()
    extract_to = '/content/_extract_tmp'
    if os.path.exists(extract_to):
        shutil.rmtree(extract_to)
    os.makedirs(extract_to)
    with zipfile.ZipFile(DRIVE_PATH, 'r') as zf:
        zf.extractall(extract_to)
    root = _find_project_root(extract_to)
    if root is None:
        raise RuntimeError(
            f'แตก zip แล้วไม่พบ audio_data/ + colab/ — ตรวจโครงสร้าง zip'
        )
    if root != WORK_DIR:
        shutil.move(root, WORK_DIR)
    if os.path.exists(extract_to):
        shutil.rmtree(extract_to, ignore_errors=True)
    print(f'✅ แตก zip เสร็จใน {time.time()-t0:.0f} วินาที')

    # 💾 สร้าง cache บน Drive ให้รอบหน้า (ทำครั้งเดียว ใช้เวลานาน ~15-30 นาที)
    if USE_CACHE and not os.path.isdir(DRIVE_CACHE):
        print(f'\n💾 กำลังสร้าง cache บน Drive ที่ {DRIVE_CACHE} ...')
        print('   (ครั้งเดียวเท่านั้น — รอบหน้าจะใช้ cache นี้แทน extract zip)')
        t0 = time.time()
        shutil.copytree(WORK_DIR, DRIVE_CACHE)
        print(f'✅ สร้าง cache เสร็จใน {(time.time()-t0)/60:.1f} นาที')
elif os.path.isdir(DRIVE_PATH):
    # ── โหมด folder: copy จาก Drive ──────────────────────────────────────
    if _find_project_root(DRIVE_PATH) is None:
        raise RuntimeError(f'{DRIVE_PATH} ไม่มี audio_data/ + colab/')
    print(f'📁 ตรวจพบโฟลเดอร์ — กำลัง copy {DRIVE_PATH} → {WORK_DIR}')
    t0 = time.time()
    shutil.copytree(DRIVE_PATH, WORK_DIR)
    print(f'✅ Copy เสร็จใน {(time.time()-t0)/60:.1f} นาที')
else:
    raise FileNotFoundError(f'ไม่พบ {DRIVE_PATH}')

os.chdir(WORK_DIR)
print(f'\n📂 ตำแหน่งงาน: {os.getcwd()}')
print(f'\nไฟล์ภายใน:')
!ls

for required in ['audio_data', 'colab']:
    if not os.path.isdir(required):
        raise RuntimeError(f'ขาดโฟลเดอร์ {required}/ — pipeline ใช้ไม่ได้')
print('\n✅ พบ audio_data/ + colab/ พร้อมเริ่ม')


## Cell 1.3 — ตรวจสอบ GPU

ดูว่ามี GPU พร้อมใช้หรือไม่ ถ้าไม่มีจะเทรนช้ามาก (ต้องไปตั้ง runtime เป็น GPU ก่อน)

**ผลลัพธ์ที่ควรเห็น:** ตาราง `nvidia-smi` แสดง `Tesla T4` หรือ `A100`


In [ ]:
!nvidia-smi


## Cell 1.4 — ติดตั้ง Python Libraries

ติดตั้ง libraries ที่จำเป็น:
- **transformers** — โมเดล Wav2Vec2
- **datasets** — โหลด dataset แบบ HuggingFace format
- **accelerate** — ใช้ GPU อย่างมีประสิทธิภาพ
- **librosa, soundfile** — อ่านไฟล์เสียง
- **evaluate, jiwer** — คำนวณ WER/CER

ใช้เวลาประมาณ 1–2 นาที


In [ ]:
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 \
    librosa soundfile evaluate jiwer


---
# 📊 ส่วนที่ 2: เตรียม Dataset

ในส่วนนี้เราจะ:
1. สร้าง **manifest** (ไฟล์ JSONL ที่บอกว่าแต่ละคลิปอยู่ที่ไหน + transcript คืออะไร)
2. สร้าง **vocabulary** (ตัวอักษรไทยทั้งหมดที่ปรากฏในข้อมูล)

## 🎯 ทำไมใช้ `build_manifest_v3.py` (Plan B — multi-dialect)?

ประวัติ version:
- `build_manifest.py` (v1): speaker split → CER ~37% เพราะ test มี utterance ที่ไม่เคยเห็น
- `build_manifest_v2.py` (v2): augmentation split สำหรับ south เท่านั้น → ไม่ทำงานกับ north/isan ที่ไม่มี `_aug_` ไฟล์
- `build_manifest_v3.py` (v3): **Plan B uniform across regions** → รองรับทั้ง 3 ภาค

v3 ทำอะไร:
- ไฟล์ `_original.wav` (เสียงสะอาด 1 ไฟล์ต่อ speaker) → **test เท่านั้น** (eval บนเสียงสะอาด)
- ไฟล์ variations อื่นๆ (child/female/male/elderly_* + south's `_aug_`) → shuffle 80/10/10 ต่อ speaker
- ทุก speaker ปรากฏใน train, val, test → closed-set evaluation

ผลลัพธ์ที่คาดหวัง: CER < 10% สำหรับทั้ง 3 ภาค (แยก eval ต่อภาคใน Cell 4.4)

## Cell 2.1 — สร้าง Manifest

รัน script `build_manifest_v3.py` ที่จะสแกน `audio_data/` ทั้ง 3 ภาค (south + north + isan) แล้วสร้าง:
- `manifests/train.jsonl` (~120,000 คลิป)
- `manifests/val.jsonl` (~15,000 คลิป)
- `manifests/test.jsonl` (~15,000 คลิป — รวมไฟล์ `_original.wav` ของทุก speaker สำหรับ eval บนเสียงสะอาด)
- `manifests/dialect_to_central.json` (lookup table 150 ประโยค: 50 ต่อภาค)

### Strategy: Plan B (สอดคล้องทุกภาค)

- `_original.wav` (เสียงสะอาด, 1 ไฟล์/speaker) → **test เท่านั้น** เพื่อ eval แบบยุติธรรม
- ไฟล์ variations อื่นๆ (child / female / male / elderly_* + south's `_aug_`) → shuffle 80/10/10 ต่อ speaker

ความแตกต่างจาก v2:
- v2 ออกแบบสำหรับ south (split `_aug_` files) — north/isan ที่ไม่มี `_aug_` จะตกใน test ทั้งหมด
- v3 treat child/female/male/elderly_* เป็น natural augmentation → ใช้ได้ทั้ง 3 ภาค


In [ ]:
print('🔄 สร้าง manifest แบบ Plan B (3 ภาค: south + north + isan) ...')
!python colab/scripts/build_manifest_v3.py
print('\n📁 ไฟล์ใน manifests/:')
!ls -la manifests/

## Cell 2.2 — ดูสถิติของ Dataset

ตรวจสอบว่า:
- จำนวนคลิปในแต่ละ split ถูกต้อง (train ≫ val ≈ test)
- มีครบทั้ง 3 ภาค (south / north / isan) ในทุก split
- Sample rate เป็น 16000 Hz ทั้งหมด
- มีครบ 150 speakers (50 ต่อภาค) ในทุก split


In [ ]:
import json
from collections import Counter

print(f"{'Split':<8}{'Clips':>10}{'Speakers':>10}{'Regions':>40}{'Duration':>14}")
print('-' * 82)
for split in ['train', 'val', 'test']:
    rows = [json.loads(l) for l in open(f'manifests/{split}.jsonl', encoding='utf-8')]
    speakers = len({(r['region'], r['speaker_id']) for r in rows})
    regions = Counter(r['region'] for r in rows)
    total_dur = sum(r['duration'] for r in rows)
    print(f"{split:<8}{len(rows):>10}{speakers:>10}{str(dict(regions)):>40}{total_dur/60:>11.1f} นาที")

# Test set composition (Plan B reserves originals to test)
print('\n— Test set category mix —')
test_rows = [json.loads(l) for l in open('manifests/test.jsonl', encoding='utf-8')]
cats = Counter(r.get('category', 'unknown') for r in test_rows)
for c, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {c:20s}: {n}")

## Cell 2.3 — สร้าง Vocabulary

สร้างไฟล์ `vocab.json` ที่บรรจุตัวอักษรไทยทั้งหมดที่ใช้ในข้อมูล + token พิเศษ:
- `|` — แทนช่องว่าง
- `[UNK]` — token ไม่รู้จัก
- `[PAD]` — blank token สำหรับ CTC


In [ ]:
!python colab/scripts/prepare_vocab.py --target text_dialect


## Cell 2.4 — ดู Vocabulary ที่สร้าง

ดูจำนวน token และตัวอักษรไทยทั้งหมด — รวม 3 ภาคจะมีประมาณ 60–70 token (ใต้ ~50, รวมเหนือ + อีสานจะมีตัวอักษรเพิ่ม)


In [ ]:
import json

vocab = json.load(open('models/vocab/vocab.json', encoding='utf-8'))
print(f'📚 จำนวน token ทั้งหมด: {len(vocab)}')

thai_chars = ''.join(k for k in vocab if len(k) == 1 and k != '|')
print(f'🔤 ตัวอักษรไทย ({len(thai_chars)} ตัว): {thai_chars}')

special = [k for k in vocab if k in ['|', '[UNK]', '[PAD]']]
print(f'🔧 Special tokens: {special}')


---
# 🏋️ ส่วนที่ 3: Train โมเดล

มี 2 cells สำหรับ train:
1. **Sanity check** (3 epochs, originals only, ~10 นาที) — ใช้ทดสอบว่า pipeline ใช้ได้
2. **Full training** (8 epochs, ~120k clips, ~5-8 ชม. T4) — เทรนจริง

### ⚡ Safe optimizations (เปิดอัตโนมัติใน script)

- `group_by_length=True` — จัด batch ให้คลิปยาวใกล้กัน → ลด padding waste
- `dataloader_pin_memory=True` — data → GPU เร็วขึ้น
- `preprocess_logits_for_metrics` — argmax บน GPU ตอน eval → ป้องกัน OOM
- `metric_for_best_model="cer"` — เหมาะกับภาษาไทย (WER ใช้ไม่ได้เพราะ Thai ไม่มี space)

### 🛡️ Reliability features (อัตโนมัติทั้งหมด)

- **Checkpoint → Google Drive** — Cell 3.2 ตั้ง `output_dir` ไปที่ Drive โดยตรง
- **Auto-resume** — รัน Cell 3.2 ใหม่ จะตรวจ checkpoint บน Drive แล้ว resume ให้เอง
- **Early stopping (patience=3)** — หยุดเองถ้า CER ไม่ดีขึ้น 3 evals ติด
- **Clean progress display** — บรรทัดเดียวต่อ logging step (step/total, %, epoch, loss trend, lr, ETA)

### ⚠️ ข้อควรรู้เรื่อง Colab ฟรี

Colab ฟรีจำกัด ~12 ชม./session — แต่ตอนนี้ training **อยู่ในช่วง 5-8 ชม.** จึงอยู่ใน budget แล้ว
ถ้ายัง disconnect: รัน Cell 1.1, 1.2, 1.3, 1.4 ใหม่ แล้วรัน Cell 3.2 — มันจะ resume เอง

Keep-alive script (optional) ใส่ใน browser console (กด F12):

```javascript
setInterval(()=>document.querySelector('colab-toolbar-button#connect')?.click(), 60000)
```


## Cell 3.1 — Sanity Check (Optional แต่แนะนำมาก)

เทรนสั้นๆ 3 epochs บน ~150 ไฟล์ original (50 speakers × 3 ภาค, ไม่รวม variations) เพื่อทดสอบว่า pipeline ทำงานได้

**สิ่งที่ควรเห็น:**
- Loss ลดลงเรื่อยๆ (เริ่มต้นประมาณ 8–10, ลงเหลือ 4–5)
- WER ออกเป็นตัวเลข (ไม่ใช่ NaN/error)

ใช้เวลา **10–15 นาที** บน T4


In [ ]:
!python colab/scripts/train_wav2vec2.py \
    --skip_augmented \
    --epochs 3 \
    --batch_size 4 \
    --eval_steps 30 \
    --save_steps 30 \
    --output_dir models/sanity-check

## Cell 3.2 — Full Training

เทรนจริงด้วย hyperparameter ที่เหมาะกับ T4 GPU + multi-dialect data:

| Parameter | ค่า | ความหมาย |
|---|---|---|
| `--epochs` | 8 | sweet spot ระหว่างเวลา/คุณภาพ (data เยอะแล้ว) |
| `--batch_size` | 16 | ใช้ VRAM ของ T4 ได้เต็มที่ (ถ้า OOM ลดเหลือ 8) |
| `--gradient_accumulation_steps` | 1 | ไม่ต้อง accumulate เพราะ batch ใหญ่พอแล้ว |
| `--lr` | 1e-4 | ปลอดภัยกว่า 3e-4 (ลดความเสี่ยง loss กระเด้ง) |
| `--warmup_steps` | 1000 | ค่อยๆ ไต่ lr ขึ้น ป้องกัน gradient explosion ช่วงแรก |
| `--eval_steps` / `--save_steps` | 500 | eval + save checkpoint ทุก 500 step |
| `--num_workers` | 2 | T4 มี CPU 2 core (เกินกว่านี้ไม่ช่วย) |
| `--output_dir` | (Drive path) | save checkpoint บน Drive โดยตรง — กัน disconnect |

**🎯 Best model:** ใช้ CER เป็นตัวตัดสิน (ไม่ใช่ WER) เพราะภาษาไทยไม่มี space แบ่งคำ

**⏱️ ระยะเวลา:** 5–8 ชั่วโมงบน T4 / 2–3 ชั่วโมงบน A100

**🔄 Resume:** ถ้า disconnect ก็แค่รัน cell ใหม่ — auto-detect checkpoint ล่าสุดบน Drive แล้ว resume ให้เอง


In [ ]:
# 📦 บันทึก checkpoint ไว้บน Google Drive โดยตรง (ไม่หายเมื่อ Colab disconnect)
import os, glob

OUTPUT_DIR = '/content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ตรวจหา checkpoint ล่าสุดอัตโนมัติ (ถ้ามี = resume; ถ้าไม่มี = start fresh)
ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'),
               key=lambda p: int(p.rsplit('-', 1)[1]))
resume_arg = f'--resume_from_checkpoint {ckpts[-1]}' if ckpts else ''

if resume_arg:
    print(f'▶️  พบ checkpoint เดิม — จะ resume จาก: {ckpts[-1]}')
else:
    print('🆕 ไม่พบ checkpoint เดิม — เริ่มเทรนใหม่ตั้งแต่ต้น')

!python colab/scripts/train_wav2vec2.py \
    --base_model airesearch/wav2vec2-large-xlsr-53-th \
    --target text_dialect \
    --epochs 8 \
    --batch_size 16 \
    --gradient_accumulation_steps 1 \
    --lr 1e-4 \
    --warmup_steps 1000 \
    --eval_steps 500 \
    --save_steps 500 \
    --num_workers 2 \
    --output_dir {OUTPUT_DIR} \
    {resume_arg}


---
# 🎤 ส่วนที่ 4: ทดสอบและประเมินผล

## Cell 4.1 — โหลดโมเดลที่เทรนเสร็จ

โหลด model + processor จากโฟลเดอร์ที่เทรนเสร็จ ขึ้นไปบน GPU


In [ ]:
import json
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import librosa

MODEL_DIR = 'models/wav2vec2-thai-dialects-v3'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).to(device)
model.eval()

print(f'✅ โมเดลพร้อมใช้บน {device}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Cell 4.2 — เตรียมฟังก์ชัน Inference

สร้างฟังก์ชัน 2 ตัว:
- `transcribe(wav_path)` — รับ path ไฟล์เสียง → return ข้อความภาษาใต้
- `to_central(text)` — รับข้อความใต้ → return ข้อความกลาง (ใช้ lookup + fuzzy match)


In [ ]:
from difflib import get_close_matches

# โหลด lookup table
lookup = json.load(open('manifests/dialect_to_central.json', encoding='utf-8'))

def transcribe(wav_path):
    """ถอดเสียงไฟล์ .wav เป็นข้อความภาษาใต้"""
    audio, _ = librosa.load(wav_path, sr=16000, mono=True)
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits
    return processor.batch_decode(torch.argmax(logits, -1))[0]

def to_central(text):
    """แปลงข้อความใต้ → กลาง โดย lookup จาก dictionary"""
    if text in lookup:
        return lookup[text]['text_central'], 'exact'
    matches = get_close_matches(text, list(lookup.keys()), n=1, cutoff=0.6)
    if matches:
        return lookup[matches[0]]['text_central'], f'fuzzy: {matches[0]}'
    return None, 'no match'

print('✅ ฟังก์ชัน transcribe() และ to_central() พร้อมใช้')


## Cell 4.3 — ทดสอบกับไฟล์ Original จาก Test Set (1 ภาคต่อ 2 ไฟล์)

ลองถอดเสียงไฟล์ `_original.wav` จากทั้ง 3 ภาค (เลือก 2 ไฟล์ต่อภาค = 6 ไฟล์) แล้วเทียบกับ ground truth


In [ ]:
# เลือก 2 ไฟล์ original จาก test set ต่อภาค
from collections import defaultdict
test_rows = [json.loads(l) for l in open('manifests/test.jsonl', encoding='utf-8')]
originals_by_region = defaultdict(list)
for r in test_rows:
    if r.get('category') in ('original', 'canonical'):
        originals_by_region[r['region']].append(r)

samples = []
for region in ['south', 'north', 'isan']:
    samples.extend(originals_by_region.get(region, [])[:2])

print('=' * 70)
for r in samples:
    pred = transcribe(r['audio_filepath'])
    central, how = to_central(pred)
    print(f"\n📁 [{r['region']}] {r['audio_filepath']}")
    print(f"  Ground truth (dialect): {r['text_dialect']}")
    print(f"  Predicted    (dialect): {pred}")
    print(f"  Translated   (central): {central}  [{how}]")
    print(f"  Expected     (central): {r['text_central']}")

## Cell 4.4 — ประเมินผลบน Test Set (WER + CER แยกตามภาค)

คำนวณตัวชี้วัดบน test set ~15,000 คลิป และแยกผลตามภาค (south / north / isan)

**ตัวชี้วัด:**
- **WER (Word Error Rate)** — % คำที่ผิด — ในภาษาไทยที่ไม่เว้นวรรค WER มักสูง
- **CER (Character Error Rate)** — % ตัวอักษรที่ผิด — ตัวชี้วัดหลักของไทย

**เกณฑ์ CER:**
- < 5% = ดีเยี่ยม
- 5–15% = ดี
- 15–25% = พอใช้
- > 25% = ต้องปรับปรุง

⚠️ ใช้เวลาประมาณ **30–40 นาที** (test set ใหญ่ขึ้น 27 เท่าจากเดิม) — ถ้าต้องการเร็ว ใส่ `SAMPLE = 500` ด้านล่าง


In [ ]:
import evaluate
from collections import defaultdict

wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')

test_rows = [json.loads(l) for l in open('manifests/test.jsonl', encoding='utf-8')]

# ถ้า test set ใหญ่เกินไป สามารถ subsample ได้
SAMPLE = None   # set to e.g. 1000 เพื่อ eval แค่ subset (เร็วขึ้นมาก)
if SAMPLE:
    import random
    random.seed(0)
    test_rows = random.sample(test_rows, min(SAMPLE, len(test_rows)))

preds_by_region = defaultdict(list)
refs_by_region = defaultdict(list)
for i, r in enumerate(test_rows):
    if i % 200 == 0:
        print(f'  {i}/{len(test_rows)} ...')
    pred = transcribe(r['audio_filepath'])
    preds_by_region[r['region']].append(pred)
    refs_by_region[r['region']].append(r['text_dialect'])

print('\n' + '=' * 60)
print(f'📊 ผลลัพธ์ (test = {len(test_rows)} คลิป)')
print('=' * 60)
all_preds, all_refs = [], []
for region in ['south', 'north', 'isan']:
    preds = preds_by_region.get(region, [])
    refs = refs_by_region.get(region, [])
    if not preds:
        continue
    wer = wer_metric.compute(predictions=preds, references=refs)
    cer = cer_metric.compute(predictions=preds, references=refs)
    print(f'[{region:>5s}] n={len(preds):>5}  WER={wer*100:>6.2f}%  CER={cer*100:>6.2f}%')
    all_preds.extend(preds); all_refs.extend(refs)

wer = wer_metric.compute(predictions=all_preds, references=all_refs)
cer = cer_metric.compute(predictions=all_preds, references=all_refs)
print('-' * 60)
print(f'[ALL ] n={len(all_preds):>5}  WER={wer*100:>6.2f}%  CER={cer*100:>6.2f}%')

---
# 💾 ส่วนที่ 5: บันทึกโมเดล

## Cell 5.1 — Save โมเดลกลับไปยัง Google Drive

⚠️ **สำคัญมาก!** ไฟล์บน Colab จะหายเมื่อ session ปิด ต้อง copy โมเดลกลับ Drive ก่อน

**ขนาดโมเดล:** ประมาณ 1.2 GB (รวม checkpoint จะใหญ่กว่านี้)


In [ ]:
import shutil
from pathlib import Path

MODEL_NAME = 'wav2vec2-thai-dialects-v3'

# ── ลบ checkpoint ก่อน save เพื่อประหยัดพื้นที่ (เก็บแค่ final model) ──
print('🗑️  ลบ checkpoint ระหว่างการเทรน ...')
!rm -rf models/{MODEL_NAME}/checkpoint-*

src = f'{WORK_DIR}/models/{MODEL_NAME}'

# ── หา destination ใน Drive: ถ้า DRIVE_PATH เป็น .zip ใช้โฟลเดอร์ข้างไฟล์ zip
#    (เพราะเขียนลง zip ตรงๆ ไม่ได้) ───────────────────────────────────────
if DRIVE_PATH.endswith('.zip'):
    drive_root = str(Path(DRIVE_PATH).parent)         # eg /content/drive/MyDrive
    dst = f'{drive_root}/{MODEL_NAME}'
    print(f'ℹ️  DRIVE_PATH เป็น zip — save โมเดลข้างไฟล์ zip ที่: {dst}')
else:
    dst = f'{DRIVE_PATH}/models/{MODEL_NAME}'

# ── ตรวจว่า src มีอยู่จริง (โมเดลเทรนเสร็จแล้ว) ──
if not os.path.isdir(src):
    raise FileNotFoundError(
        f'ไม่พบโมเดลที่ {src} — รัน Cell 3.2 ให้เทรนเสร็จก่อน'
    )

print(f'\n📤 กำลัง copy {src} → {dst}')
os.makedirs(os.path.dirname(dst), exist_ok=True)   # parent ของ dst ต้องมีก่อน
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print('\n✅ Save เสร็จสิ้น ขนาดโมเดล:')
!du -sh "$dst"

---
# 🔬 ส่วนที่ 6: ทดสอบกับเสียงของคุณเอง (Optional)

## Cell 6.1 — อัพโหลดไฟล์เสียงทดสอบ

อัพโหลดไฟล์ `.wav` ใดๆ เพื่อทดสอบ — แนะนำให้เป็นเสียงภาษาถิ่น (ใต้/เหนือ/อีสาน) ที่พูด 1 ใน 150 ประโยคที่อยู่ใน lookup table

**คำแนะนำ:** ไฟล์ควรเป็น 16kHz mono ความยาว 1–4 วินาที


In [ ]:
from google.colab import files

print('📤 เลือกไฟล์ .wav ที่จะทดสอบ:')
uploaded = files.upload()

print('\n' + '=' * 70)
for filename in uploaded:
    pred = transcribe(filename)
    central, how = to_central(pred)
    print(f"\n📁 {filename}")
    print(f"  Dialect : {pred}")
    print(f"  Central : {central}  [{how}]")


---
# 📦 ส่วนที่ 7: โหลดโมเดลที่เทรนเสร็จแล้วเพื่อใช้ต่อ

ใช้เมื่อ:
- กลับมาเปิด Colab session ใหม่หลังจากเทรนเสร็จไปแล้ว (ไม่ต้องเทรนซ้ำ)
- ย้ายไปรันบนเครื่อง local
- โหลดโมเดลของคนอื่นจาก Hugging Face Hub มาทดสอบ
- โหลดโมเดลจากไฟล์ `.zip` (เช่นที่ดาวน์โหลดมาจาก Google Drive ผ่านเว็บ — Drive จะ zip ให้อัตโนมัติ)

ฟังก์ชัน `load_trained_model()` ด้านล่างรองรับ **4 รูปแบบ path**:

| รูปแบบ | ตัวอย่าง path |
|---|---|
| Google Drive folder | `/content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3` |
| Local folder | `models/wav2vec2-thai-dialects-v3` หรือ `/path/to/model` |
| Hugging Face Hub | `username/wav2vec2-thai-dialects-v3` |
| ไฟล์ `.zip` | `/content/wav2vec2-thai-dialects-v3.zip` (จะ extract ให้อัตโนมัติ) |

> ⚠️ ก่อนรัน cell นี้ต้องติดตั้ง libraries แล้ว (Cell 1.4) และ mount Drive ไว้แล้ว (Cell 1.1) ถ้าจะโหลดจาก Drive

In [ ]:
import os
import json
import shutil
import zipfile
import tempfile
import torch
import librosa
from pathlib import Path
from difflib import get_close_matches
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor


def _find_model_dir(root: str) -> str:
    """หา folder ที่มี config.json ภายใน (สำหรับกรณี zip มี nested folder)"""
    root_path = Path(root)
    if (root_path / 'config.json').exists():
        return str(root_path)
    # search 2 levels deep
    for cfg in root_path.rglob('config.json'):
        # skip nested checkpoint folders
        if 'checkpoint-' in str(cfg.parent.name):
            continue
        return str(cfg.parent)
    raise FileNotFoundError(f'ไม่พบ config.json ใน {root} — โครงสร้างโมเดลไม่ถูกต้อง')


def _maybe_extract_zip(model_path: str, extract_dir: str = None) -> str:
    """ถ้า model_path เป็นไฟล์ .zip ให้ extract แล้วคืน path ของ folder โมเดล"""
    if not model_path.endswith('.zip'):
        return model_path
    if not os.path.exists(model_path):
        raise FileNotFoundError(f'ไม่พบไฟล์ zip: {model_path}')

    if extract_dir is None:
        # default: extract ลงโฟลเดอร์ข้างไฟล์ zip
        extract_dir = model_path.removesuffix('.zip') + '_extracted'

    if os.path.isdir(extract_dir) and os.listdir(extract_dir):
        print(f'✓ ใช้โฟลเดอร์ที่ extract ไว้แล้ว: {extract_dir}')
    else:
        os.makedirs(extract_dir, exist_ok=True)
        print(f'📦 กำลัง extract {model_path} → {extract_dir} ...')
        with zipfile.ZipFile(model_path, 'r') as zf:
            zf.extractall(extract_dir)
        print(f'✅ extract เสร็จ ({len(os.listdir(extract_dir))} รายการ)')

    return _find_model_dir(extract_dir)


def load_trained_model(model_path: str, device: str = 'auto', extract_dir: str = None):
    """โหลดโมเดล Wav2Vec2 ที่เทรนเสร็จแล้ว เพื่อนำไปใช้ inference

    รองรับ 4 รูปแบบ path:
        1. Google Drive: '/content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3'
        2. Local folder: 'models/wav2vec2-thai-dialects-v3' หรือ absolute path
        3. HF Hub:       'username/repo-name'
        4. .zip file:    'path/to/model.zip' (จะ extract ให้อัตโนมัติ)

    Args:
        model_path: path ไปยังโฟลเดอร์โมเดล / repo id / ไฟล์ .zip
        device: 'auto' | 'cuda' | 'cpu'
        extract_dir: โฟลเดอร์ปลายทางสำหรับ extract zip (default: ข้าง zip)

    Returns:
        (model, processor) ที่พร้อมใช้ inference
    """
    if device == 'auto':
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ── ถ้าเป็น zip ก็ extract ก่อน
    resolved_path = _maybe_extract_zip(model_path, extract_dir)

    # ── ตรวจประเภท source (สำหรับแสดงผล)
    if model_path.endswith('.zip'):
        src = f'Zip → {resolved_path}'
    elif os.path.exists(resolved_path):
        src = 'Local/Drive'
    else:
        src = 'Hugging Face Hub'

    print(f'⏳ กำลังโหลดโมเดลจาก {src}: {resolved_path}')

    try:
        processor = Wav2Vec2Processor.from_pretrained(resolved_path)
        model = Wav2Vec2ForCTC.from_pretrained(resolved_path).to(device)
        model.eval()
    except Exception as e:
        raise RuntimeError(
            f'โหลดโมเดลไม่สำเร็จ: {e}\n'
            f'  • ถ้าโหลดจาก Drive — ตรวจว่า mount Drive แล้วและ path ถูกต้อง\n'
            f'  • ถ้าโหลดจาก Local — ตรวจว่ามีไฟล์ config.json ในโฟลเดอร์\n'
            f'  • ถ้าโหลดจาก zip — ตรวจว่าไฟล์ zip ไม่เสียและมีโครงสร้างที่ถูก\n'
            f'  • ถ้าโหลดจาก HF Hub — ตรวจ repo id หรือ login ด้วย huggingface-cli login'
        ) from e

    n_params = sum(p.numel() for p in model.parameters())
    print(f'✅ โหลดสำเร็จ — {n_params:,} parameters บน {device}')
    return model, processor


def make_transcribe_fn(model, processor, device=None):
    """สร้างฟังก์ชัน transcribe(wav_path) ที่ผูกกับ model + processor นี้"""
    if device is None:
        device = next(model.parameters()).device

    def transcribe(wav_path: str) -> str:
        audio, _ = librosa.load(wav_path, sr=16000, mono=True)
        inputs = processor(audio, sampling_rate=16000, return_tensors='pt')
        with torch.no_grad():
            logits = model(inputs.input_values.to(device)).logits
        return processor.batch_decode(torch.argmax(logits, -1))[0]

    return transcribe


def load_dialect_lookup(lookup_path: str):
    """โหลด lookup table สำหรับแปลงภาษาถิ่น → ภาษากลาง"""
    lookup = json.load(open(lookup_path, encoding='utf-8'))

    def to_central(text: str):
        if text in lookup:
            return lookup[text]['text_central'], 'exact'
        matches = get_close_matches(text, list(lookup.keys()), n=1, cutoff=0.6)
        if matches:
            return lookup[matches[0]]['text_central'], f'fuzzy: {matches[0]}'
        return None, 'no match'

    return to_central


print('✅ ฟังก์ชันพร้อมใช้: load_trained_model() (รองรับ .zip), make_transcribe_fn(), load_dialect_lookup()')

## Cell 7.2 — ตัวอย่างการเรียกใช้

เลือก path ตามที่เก็บโมเดลไว้ (ปลด comment เฉพาะรูปแบบที่ใช้):

```python
# 1. โหลดจาก Google Drive folder (กรณีกลับมาเปิด session ใหม่)
MODEL_PATH = '/content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3'

# 2. โหลดจาก local Colab disk (กรณีเพิ่งเทรนเสร็จใน session นี้)
# MODEL_PATH = 'models/wav2vec2-thai-dialects-v3'

# 3. โหลดจาก Hugging Face Hub
# MODEL_PATH = 'your-username/wav2vec2-thai-dialects-v3'

# 4. โหลดจากไฟล์ .zip (เช่นโหลดจาก Drive ผ่านเว็บ → ได้เป็น zip)
# MODEL_PATH = '/content/wav2vec2-thai-dialects-v3.zip'
# (จะ extract ให้อัตโนมัติที่ '/content/wav2vec2-thai-dialects-v3_extracted')
```

In [ ]:
# กำหนด path ของโมเดล (แก้ตามที่เก็บไว้)
MODEL_PATH = '/content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3'

# (Optional) path ไปยัง lookup table สำหรับแปลงเป็นภาษากลาง
LOOKUP_PATH = '/content/drive/MyDrive/Data_Project/manifests/dialect_to_central.json'

# 1. โหลดโมเดล
model, processor = load_trained_model(MODEL_PATH)

# 2. สร้างฟังก์ชัน transcribe ผูกกับโมเดลที่โหลด
transcribe = make_transcribe_fn(model, processor)

# 3. (Optional) โหลด lookup table สำหรับแปลงภาษา
to_central = load_dialect_lookup(LOOKUP_PATH) if os.path.exists(LOOKUP_PATH) else None

# 4. ทดสอบกับไฟล์เสียง 1 ไฟล์ (ปลด comment + แก้ path)
# WAV_PATH = '/content/drive/MyDrive/Data_Project/audio_data/south/voice_1/voice_1_1_0000_original.wav'
# WAV_PATH = '/content/drive/MyDrive/Data_Project/audio_data/north/1/1._0000_original.wav'
# WAV_PATH = '/content/drive/MyDrive/Data_Project/audio_data/isan/1/1_0000_original.wav'
# pred = transcribe(WAV_PATH)
# print(f'Dialect: {pred}')
# if to_central:
#     central, how = to_central(pred)
#     print(f'Central: {central}  [{how}]')

print('\n💡 ปลด comment ตัวอย่างด้านล่างและใส่ path ไฟล์ .wav เพื่อทดสอบ')

---
# 🆘 Troubleshooting

## ปัญหาที่พบบ่อย

| อาการ | วิธีแก้ |
|---|---|
| `CUDA out of memory` | ลด `--batch_size` เหลือ 8 (หรือ 4) — ถ้ายังไม่พอ เพิ่ม `--gradient_accumulation_steps 2` |
| `Loss = NaN` | ลด `--lr` เหลือ `5e-5` |
| `CER ไม่ลด` | เช็คว่า vocab สร้างถูก (Cell 2.4), ลอง sanity check (Cell 3.1) ก่อน |
| `Disk เต็ม` (Drive) | ลบ checkpoint เก่า: `!rm -rf /content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3/checkpoint-*` |
| Cell 1.2 copy ช้ามาก | ครั้งแรกจะนาน (สร้าง cache) — รอบหน้าจะเร็วขึ้นมาก |
| `config.json not found` ตอน inference | โมเดลยังไม่เทรนเสร็จ — รัน Cell 3.2 ให้จบก่อน |
| Colab disconnect ระหว่างเทรน | **ไม่ต้องกังวล** — รัน Cell 1.1-1.4 + Cell 3.2 ใหม่ จะ auto-resume จาก Drive |
| ไฟล์ `_aug_` ไม่อยู่ใน north/isan | ปกติ — Plan B treat child/female/male/elderly_* เป็น augmentation แทน |
| Training หยุดเองก่อนครบ epoch | **ปกติ** — early stopping (patience=3) หยุดเมื่อ CER ไม่ดีขึ้น 3 evals ติด |

## การ Resume Training (อัตโนมัติแล้ว)

**ไม่ต้องระบุ `--resume_from_checkpoint` เอง** — Cell 3.2 มีโค้ดตรวจ checkpoint ล่าสุดบน Drive แล้วส่ง argument ให้ script เอง

ถ้าอยาก resume แบบ manual จาก checkpoint เฉพาะเจาะจง:

```bash
!python colab/scripts/train_wav2vec2.py \\
    --base_model airesearch/wav2vec2-large-xlsr-53-th \\
    --target text_dialect \\
    --epochs 8 \\
    --batch_size 16 \\
    --lr 1e-4 \\
    --num_workers 2 \\
    --output_dir /content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3 \\
    --resume_from_checkpoint /content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3/checkpoint-XXXX
```

(แทน `XXXX` ด้วยเลข checkpoint ล่าสุด — ดูจาก `!ls /content/drive/MyDrive/Data_Project/models/wav2vec2-thai-dialects-v3/`)

## เคล็ดลับ Colab ฟรี

- **Disconnect:** ฟรีจะตัดหลัง 90 นาที (ไม่มี activity) และจำกัด ~12 ชม. ต่อ session
- **Auto-resume + Drive cache** ทำให้ disconnect ไม่ใช่ปัญหาใหญ่อีกต่อไป — แค่รัน cells เตรียมตัวใหม่ก็พอ
- **Keep-alive** (optional) ใส่ JavaScript ใน browser console:
  ```javascript
  setInterval(()=>document.querySelector('colab-toolbar-button#connect')?.click(), 60000)
  ```
